In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import time 

C:\Users\PHU\AppData\Local\Temp\ipykernel_23092\1439919409.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
# Define the URL
url = "https://api.tomorrow.io/v4/weather/realtime?location=10.7829647,106.670745&apikey=Ez2hSkCFqMrvsXGs56HWnk7eWcKGwGP8"

In [3]:
# Initialize an empty list to store the responses
dataframes = []

# Function to call the API, add response data to list of dataframes, and save to CSV
def call_api_and_save():
    try:
        headers = {"accept": "application/json"}
        response = requests.get(url, headers=headers)
        response_data = response.json()
        print(response_data)
        # Assuming the response contains a 'data' field with relevant information
        if 'data' in response_data:
            # Convert response data to dataframe
            new_data = pd.DataFrame(response_data)
            
            # Append dataframe to the list
            dataframes.append(new_data)
            
            # Concatenate all dataframes in the list
            combined_dataframe = pd.concat(dataframes, ignore_index=True)
            
            # Save combined dataframe to CSV
            combined_dataframe.to_csv('../data/weather data/weather_data_3-3.csv', index=False)
            
            print("Data saved successfully.")
        else:
            print("No data found in response.")
    except Exception as e:
        print("Error occurred:", e)

# Call the API and save data every 15 minutes
for i in range(30):
    call_api_and_save()
    time.sleep(900)  # 900 seconds = 15 minutes

{'data': {'time': '2024-03-03T15:16:00Z', 'values': {'cloudBase': 0.77, 'cloudCeiling': 0.77, 'cloudCover': 84, 'dewPoint': 22, 'freezingRainIntensity': 0, 'humidity': 70, 'precipitationProbability': 0, 'pressureSurfaceLevel': 1011.98, 'rainIntensity': 0, 'sleetIntensity': 0, 'snowIntensity': 0, 'temperature': 28, 'temperatureApparent': 30.67, 'uvHealthConcern': 0, 'uvIndex': 0, 'visibility': 9.99, 'weatherCode': 1001, 'windDirection': 140, 'windGust': 8.81, 'windSpeed': 5.69}}, 'location': {'lat': 10.7829647, 'lon': 106.670745}}
Data saved successfully.
{'data': {'time': '2024-03-03T15:31:00Z', 'values': {'cloudBase': 0.77, 'cloudCeiling': 0.77, 'cloudCover': 85, 'dewPoint': 22, 'freezingRainIntensity': 0, 'humidity': 70, 'precipitationProbability': 0, 'pressureSurfaceLevel': 1011.99, 'rainIntensity': 0, 'sleetIntensity': 0, 'snowIntensity': 0, 'temperature': 28, 'temperatureApparent': 30.67, 'uvHealthConcern': 0, 'uvIndex': 0, 'visibility': 9.99, 'weatherCode': 1001, 'windDirection':

In [2]:
url = "https://api.tomorrow.io/v4/timelines?apikey=Ez2hSkCFqMrvsXGs56HWnk7eWcKGwGP8"

In [3]:
payload = {
    "location": "10.7829647,106.670745",
    "fields": ["temperature", "cloudBase", "cloudCeiling", "cloudCover", "dewPoint", "freezingRainIntensity", "humidity", "precipitationProbability", "pressureSurfaceLevel", "rainIntensity", "sleetIntensity", "snowIntensity", "temperatureApparent", "uvHealthConcern", "uvIndex", "visibility", "weatherCode", "windDirection", "windGust", "windSpeed"],
    "units": "metric",
    "timesteps": ["15m"],
    "startTime": "nowMinus24h",
    "endTime": "now",
    "timezone": "auto"
}
headers = {
    "accept": "application/json",
    "Accept-Encoding": "gzip",
    "content-type": "application/json"
}

In [4]:
response = requests.post(url, json=payload, headers=headers)

print(response.text)

{"data":{"timelines":[{"timestep":"15m","endTime":"2024-03-23T00:07:00+07:00","startTime":"2024-03-22T00:07:00+07:00","intervals":[{"startTime":"2024-03-22T00:07:00+07:00","values":{"cloudBase":0.77,"cloudCeiling":null,"cloudCover":44,"dewPoint":22,"freezingRainIntensity":0,"humidity":70,"precipitationProbability":5,"pressureSurfaceLevel":1010.85,"rainIntensity":0.15,"sleetIntensity":0,"snowIntensity":0,"temperature":28,"temperatureApparent":30.67,"uvHealthConcern":0,"uvIndex":0,"visibility":9.99,"weatherCode":1101,"windDirection":150,"windGust":5.85,"windSpeed":4.13}},{"startTime":"2024-03-22T00:22:00+07:00","values":{"cloudBase":0.77,"cloudCeiling":null,"cloudCover":44,"dewPoint":22,"freezingRainIntensity":0,"humidity":70,"precipitationProbability":5,"pressureSurfaceLevel":1010.67,"rainIntensity":0.25,"sleetIntensity":0,"snowIntensity":0,"temperature":28,"temperatureApparent":30.67,"uvHealthConcern":0,"uvIndex":0,"visibility":9.99,"weatherCode":1101,"windDirection":150,"windGust":5.7

In [5]:
# Convert response data to dataframe and save to CSV
response_data = response.json()
new_data = pd.DataFrame(response_data)
#separate each start time to a row
new_data = new_data.explode('data')
#separate each data to a column
new_data = pd.concat([new_data.drop(['data'], axis=1), new_data['data'].apply(pd.Series)], axis=1)
new_data.head()

# Save combined dataframe to json
new_data.to_json('../data/weather data/weather_data_22-3.json', orient='records')

In [22]:
#apply pd.Series to intervals column, turn it into a dataframe
intervals = new_data['intervals'].apply(pd.Series)

In [23]:
intervals.head()

,0,1,2,3,4,5,6,7,8,9,...,87,88,89,90,91,92,93,94,95,96
timelines,"{'startTime': '2024-03-15T22:44:00+07:00', 'va...","{'startTime': '2024-03-15T22:59:00+07:00', 'va...","{'startTime': '2024-03-15T23:14:00+07:00', 'va...","{'startTime': '2024-03-15T23:29:00+07:00', 'va...","{'startTime': '2024-03-15T23:44:00+07:00', 'va...","{'startTime': '2024-03-15T23:59:00+07:00', 'va...","{'startTime': '2024-03-16T00:14:00+07:00', 'va...","{'startTime': '2024-03-16T00:29:00+07:00', 'va...","{'startTime': '2024-03-16T00:44:00+07:00', 'va...","{'startTime': '2024-03-16T00:59:00+07:00', 'va...",...,"{'startTime': '2024-03-16T20:29:00+07:00', 'va...","{'startTime': '2024-03-16T20:44:00+07:00', 'va...","{'startTime': '2024-03-16T20:59:00+07:00', 'va...","{'startTime': '2024-03-16T21:14:00+07:00', 'va...","{'startTime': '2024-03-16T21:29:00+07:00', 'va...","{'startTime': '2024-03-16T21:44:00+07:00', 'va...","{'startTime': '2024-03-16T21:59:00+07:00', 'va...","{'startTime': '2024-03-16T22:14:00+07:00', 'va...","{'startTime': '2024-03-16T22:29:00+07:00', 'va...","{'startTime': '2024-03-16T22:44:00+07:00', 'va..."


In [35]:
#for each column in intervals, there are 'startTime' and 'values' columns, with each value of startTime has a corresponding value in 'values'
start_time = pd.DataFrame()
values = pd.DataFrame()
for column in intervals.columns:
    start_time[column] = intervals[column].apply(lambda x: x['startTime'])
    values[column] = intervals[column].apply(lambda x: x['values'])


values = values.apply(lambda x: x.apply(pd.Series))


ValueError: If using all scalar values, you must pass an index

In [34]:
values.head()

,0,1,2,3,4,5,6,7,8,9,...,87,88,89,90,91,92,93,94,95,96
timelines,"{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.39, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.26, 'cloudCeiling': None, 'clo...",...,"{'cloudBase': None, 'cloudCeiling': None, 'clo...","{'cloudBase': None, 'cloudCeiling': None, 'clo...","{'cloudBase': None, 'cloudCeiling': None, 'clo...","{'cloudBase': 1.03, 'cloudCeiling': None, 'clo...","{'cloudBase': 1.03, 'cloudCeiling': None, 'clo...","{'cloudBase': 1.03, 'cloudCeiling': None, 'clo...","{'cloudBase': 1.03, 'cloudCeiling': None, 'clo...","{'cloudBase': 1.03, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.89, 'cloudCeiling': None, 'clo...","{'cloudBase': 0.89, 'cloudCeiling': None, 'clo..."
